# 11 — Algorithms, Complexity, and Cache Design

Explore algorithmic trade-offs, search, heap-based priority queues, bounded caching, and the consequences of stale or incorrectly keyed results.

*Advanced · 90–150 min*

## You will learn

- Derive basic complexity estimates
- Use bisect and heapq appropriately
- Apply lru_cache to pure functions
- Design invalidation and bounded cache policies

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Binary search preconditions

bisect assumes a sorted sequence and locates an insertion point in logarithmic comparisons, though insertion into a Python list still requires shifting elements. Always verify the sorted-order precondition.

In [ ]:
from bisect import bisect_left
values = [2, 4, 7, 11, 15]
index = bisect_left(values, 7)
print(index, values[index])
assert index == 2

**Design review:** Would searching an unsorted list with bisect produce trustworthy results?

### 2. Heaps and priorities

heapq implements a min-heap: the smallest item is returned first. Use tuples for stable tie-breakers such as increasing sequence numbers; otherwise incomparable objects may be compared when priorities tie.

In [ ]:
import heapq
tasks = []
heapq.heappush(tasks, (2, 0, "email"))
heapq.heappush(tasks, (1, 1, "backup"))
heapq.heappush(tasks, (2, 2, "report"))
print([heapq.heappop(tasks)[2] for _ in range(len(tasks))])

**Design review:** Why include a second numeric field if two items have equal priority?

### 3. Memoization and purity

functools.lru_cache avoids repeated pure computation for the same hashable inputs. Because it retains results, a bounded maxsize is often appropriate. Never cache results that depend on hidden mutable state without explicit invalidation.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(24))
print(fibonacci.cache_info())

**Design review:** How could caching a current exchange rate return an incorrect value later?

### 4. Complexity of transformations

Dedupe with set membership provides expected linear-time construction, while sorting adds O(n log n) time. If the input order matters, use an ordered result list and a separate membership set.

In [ ]:
def ordered_unique(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

assert ordered_unique(["b", "a", "b", "c"]) == ["b", "a", "c"]
print(ordered_unique([2, 1, 2, 3]))

**Design review:** What constraints does set membership place on item hashability?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_security_testing_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/11', {1: '2efd34f696b122aa', 2: '19a923a6558d6dda', 3: 'b15353bb78222c12', 4: 'a0bdb44d73867e16'})

**Exercise 1.** Count how many comparisons a linear search needs in the worst case.

In [ ]:
def linear_search_steps(values, target):
    steps = 0
    for v in values:
        steps += 1
        if v == target:
            break
    return steps

check(1, [linear_search_steps(list(range(100)), ___), linear_search_steps(list(range(100)), 0)])

**Exercise 2.** Binary search with `bisect`; return -1 when the value is missing.

In [ ]:
import bisect

def find(sorted_values, target):
    i = bisect.___(sorted_values, target)
    return i if i < len(sorted_values) and sorted_values[i] == target else -1

check(2, [find([1, 3, 5, 7], 5), find([1, 3, 5, 7], 4), find([], 1)])

**Exercise 3.** The three smallest values with `heapq` without sorting everything.

In [ ]:
import heapq
check(3, heapq.___(3, [9, 4, 7, 1, 8, 2]))

**Exercise 4.** Cache a pure function with `lru_cache` and check the cache was used.

In [ ]:
import functools

@functools.lru_cache(maxsize=___)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib.cache_clear()
value = fib(60)
check(4, (value, fib.cache_info().hits > 0))

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** bisect works on…

   - **a)** Any list
   - **b)** Sets
   - **c)** A sorted list

**2.** heapq.heappop returns…

   - **a)** The largest item
   - **b)** The smallest item
   - **c)** The last pushed item

**3.** The hard part of caching changing data is…

   - **a)** Syntax
   - **b)** Hashing
   - **c)** Invalidation

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/11', {1: '9877578a831b', 2: 'a149e53f7ccc', 3: 'dc9d7af7573c'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [12 — Architecture, Ports and Adapters, Dependency Injection](12_architecture_dependency_injection.ipynb)